# 📊 Đánh Giá và So Sánh Động (Dynamic Benchmark)

In [ ]:
# 1. KẾT NỐI GOOGLE DRIVE & DI CHUYỂN VÀO DỰ ÁN
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = "/content/drive/MyDrive/COMPUTER SCIENCE/NAM3_HK3_(2025-2026)/CT282_Deep Learning/PROJECT/RIFE-MinhTri/RIFE-Project"
os.chdir(PROJECT_DIR)
print(f"✅ Đang ở thư mục dự án: {os.getcwd()}")

In [ ]:
# 2. ĐỌC KẾT QUẢ VÀ VẼ BIỂU ĐỒ SO SÁNH TỰ ĐỘNG
import os
import json
import matplotlib.pyplot as plt
import pandas as pd

models_dir = "trained_model"
results = {}

# Quét tất cả các folder thí nghiệm
if os.path.exists(models_dir):
    for model_name in os.listdir(models_dir):
        json_file = os.path.join(models_dir, model_name, "experiment_results.json")
        if os.path.exists(json_file):
            with open(json_file, "r") as f:
                data = json.load(f)
                if isinstance(data, list):
                    results[model_name] = [item["val_psnr"] for item in data]
                elif isinstance(data, dict) and "history" in data:
                    results[model_name] = data["history"]["val_psnr"]

if not results:
    print("⚠️ Chưa tìm thấy kết quả thí nghiệm trong trained_model/. Hãy chạy train ít nhất 1 mô hình.")
else:
    print(f"✅ Đã nạp kết quả của {len(results)} mô hình: {list(results.keys())}")

    # Vẽ biểu đồ PSNR
    plt.figure(figsize=(10, 6), dpi=150)
    for model_name, psnr_list in results.items():
        if "baseline" in model_name:
            plt.plot(psnr_list, label=f"Baseline (PReLU)", color="black", linestyle="--", linewidth=2)
        else:
            plt.plot(psnr_list, label=model_name, linewidth=1.5)

    plt.title("So Sánh Đường Cong Hội Tụ Val PSNR (40 Epochs)", fontsize=14, fontweight="bold")
    plt.xlabel("Epoch", fontsize=12)
    plt.ylabel("PSNR (dB)", fontsize=12)
    plt.legend()
    plt.grid(True, linestyle=":", alpha=0.6)
    plt.show()

    # Bảng tổng kết so sánh với Baseline
    baseline_key = next((k for k in results.keys() if "baseline" in k), None)
    baseline_max = max(results[baseline_key]) if baseline_key else None

    summary = []
    for model_name, psnr_list in results.items():
        best_psnr = max(psnr_list)
        if baseline_max is not None and model_name != baseline_key:
            delta = best_psnr - baseline_max
            delta_str = f"{delta:+.2f} dB"
        elif model_name == baseline_key:
            delta_str = "Mốc chuẩn (0.00)"
        else:
            delta_str = "N/A"
            
        summary.append({
            "Mô Hình": model_name,
            "Best PSNR (dB)": f"{best_psnr:.2f}",
            "Chênh lệch (Δ PSNR)": delta_str
        })

    df = pd.DataFrame(summary)
    print("
📋 BẢNG TỔNG KẾT SO SÁNH HIỆU NĂNG:")
    display(df)